# Week 10 Problem Set: The List

You are the field director. Your volunteers can knock about 9,000 doors in a district of 60,000 registered voters, and the campaign bought a turnout file from Keystone Analytics.

Four tasks: six written questions and one memo.

**Lying with data, the checklist so far:**
1. **W1:** Conflating fixed and marginal costs.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as a causal effect.
7. **W8:** Cherry-picking polls; house effects; ignoring nonresponse bias.
8. **W9:** Ranking tactics by effect size or cost per attempt instead of cost per vote.
9. **W10:** An accuracy number quoted with no floor beside it, a model graded on the data it was built from, and a model of **who votes** sold as a model of **who you can move**.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the voter file straight from the course repository; you just need to be online.

Stuck? See the **Colab guide**, linked from the syllabus.

## Setup

Same split as in class: build on 42,000 voters, test on the other 18,000.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

vf = pd.read_csv(
    'https://raw.githubusercontent.com/joshuakalla/'
    'data_science_campaigns_26/main/weeks/'
    'wk10_turnout_models/data/district_voter_file.csv')

# Pick 70% of voters at random to build on. random_state=10
# fixes the random draw, so everyone gets the same 42,000.
train = vf.sample(frac=0.7, random_state=10)

# Drop those 42,000. The 18,000 left are held back for testing.
test = vf.drop(train.index)
print(len(train), 'to build on,', len(test), 'held back')

## Task 1: The free answer

Keystone says its model classified 80.0% of registered voters correctly against the 2024 general election.

The **free answer** is the accuracy of writing the more common answer on every row. Fill in the two blanks.

*Check:* the 2024 free answer is 75.9% and the 2022 free answer is 50.9%.

In [ ]:
# Keystone calls anyone scored 0.5 or above a voter.
# astype(int) turns True/False into 1/0.
keystone_call = (test['keystone_score'] >= 0.5).astype(int)
acc_2024 = (keystone_call == test['voted_2024']).mean()

# Turnout rates
t24 = test['voted_2024'].mean()
t22 = test['voted_2022'].mean()

# The free answer is whichever is bigger: the share who
# voted, or the share who did not. max(a, b) gives the
# larger of a and b.
free_2024 = # YOUR CODE HERE

free_2022 = # YOUR CODE HERE

print('Keystone vs 2024:', round(100 * acc_2024, 1))
print('free answer 2024:', round(100 * free_2024, 1))
print('free answer 2022:', round(100 * free_2022, 1))

### Question 1

In two sentences: how far above the free answer is Keystone's model in 2024, and why does 80.0% look better than it is?

**Your answer:**

*Replace this text with your answer.*

### Question 2

Turnout on this file in 2014 was 41.1%. What is the free answer for 2014, and which answer gets it?

**Your answer:**

*Replace this text with your answer.*

## Task 2: Which mistakes

Build the midterm model, then count the four boxes of its confusion matrix.

*Check:* TP 7,526, FP 1,717, FN 1,645, TN 7,112. They sum to 18,000. Accuracy is 81.3%, precision 81.4%, recall 82.1%.

In [ ]:
X = 'voted_2012 + voted_2014 + voted_2016 + voted_2018 + age'
mid = smf.ols('voted_2022 ~ ' + X, data=train).fit()

# predict() plugs each voter's history and age into the model
test['our_score'] = mid.predict(test)
our_call = (test['our_score'] >= 0.5).astype(int)

# & means both must be true. sum() counts the people where
# it is. tp is done for you; write the other three.
tp = ((our_call == 1) & (test['voted_2022'] == 1)).sum()
fp = # YOUR CODE HERE
fn = # YOUR CODE HERE
tn = # YOUR CODE HERE

print('TP', tp, ' FP', fp, ' FN', fn, ' TN', tn,
      ' total', tp + fp + fn + tn)
print('accuracy ', round(100 * (tp + tn) / len(test), 1))
print('precision', round(100 * tp / (tp + fp), 1))
print('recall   ', round(100 * tp / (tp + fn), 1))

### Question 3

Keystone's recall against 2024 is 95.2%, higher than our model's 82.1%. In three sentences, explain why that does not make Keystone the better model. What would a model's recall be if it called every single person a voter?

**Your answer:**

*Replace this text with your answer.*

## Task 3: Calibration, and where the model breaks

The same two tables as in class: our model, then Keystone's, each tested on 2022.

*Check:* in our model's table, the worst range is the bottom one: average score 0.085, and 0.042 voted.

In [ ]:
# pd.cut puts each voter's score into one of ten ranges
ranges = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
test['our_range'] = pd.cut(test['our_score'], ranges)

# For each range: the average score, and the share who
# voted. observed=True skips ranges with nobody in them.
test.groupby('our_range', observed=True)[
    ['our_score', 'voted_2022']].mean().round(3)

In [ ]:
test['keystone_range'] = pd.cut(test['keystone_score'], ranges)
test.groupby('keystone_range', observed=True)[
    ['keystone_score', 'voted_2022']].mean().round(3)

### Question 4

Compare the two tables. In one sentence, say what is wrong with Keystone's scores. Would subtracting the same number from every Keystone score fix it?

**Your answer:**

*Replace this text with your answer.*

Now do the same comparison by age band. Write it from scratch, using the same pattern as the two cells above.

*Check:* the youngest band, (19, 25], has an average score of 0.122, and 0.239 voted. It is the worst band by a wide margin.

In [ ]:
test['age_band'] = pd.cut(test['age'],
                          [19, 25, 34, 44, 54, 64, 91])

# Group test by age_band. For each band, report the average
# of 'our_score' and the average of 'voted_2022'.
by_age = # YOUR CODE HERE

by_age.round(3)

Run the cell below before you answer Question 5.

In [ ]:
# Everyone with a 0 in all four vote-history columns
never = vf[(vf['voted_2012'] == 0) & (vf['voted_2014'] == 0)
           & (vf['voted_2016'] == 0) & (vf['voted_2018'] == 0)]

# Age in this file is age in 2026, so anyone under 26 was
# under 18 in 2018. Split at 26: who voted in 2022?
young = never[never['age'] < 26]
older = never[never['age'] >= 26]
print('under 26:   ', len(young), 'people,',
      round(100 * young['voted_2022'].mean(), 1), '% voted')
print('26 and over:', len(older), 'people,',
      round(100 * older['voted_2022'].mean(), 1), '% voted')

### Question 5

Name the age band the model gets most wrong, and say whether it scores them too high or too low. Then explain **why**, using the output above.

**Your answer:**

*Replace this text with your answer.*

## Task 4: The memo

Run the cell below. It scores everyone for 2026 (Step 3), builds the three lists, and shows the party registration of everyone in Katie's range.

In [ ]:
# Move each voter's history forward four years. predict()
# looks up columns by name, so 2016 history goes in the
# voted_2012 column, 2018 in voted_2014, and so on.
moved_forward = pd.DataFrame({
    'voted_2012': vf['voted_2016'],
    'voted_2014': vf['voted_2018'],
    'voted_2016': vf['voted_2020'],
    'voted_2018': vf['voted_2022'],
    'age': vf['age'] + 4})
vf['score_2026'] = mid.predict(moved_forward)

rachel = vf.nlargest(9000, 'score_2026')    # 9,000 highest
marcus = vf.nsmallest(9000, 'score_2026')   # 9,000 lowest
# Katie: 9,000 picked at random from everyone scored
# 0.35 to 0.65
middle = vf[vf['score_2026'].between(0.35, 0.65)]
katie = middle.sample(9000, random_state=3)

# Average 2026 score: the share expected to vote if
# nobody knocks
print('Rachel:', round(100 * rachel['score_2026'].mean(), 1))
print('Katie: ', round(100 * katie['score_2026'].mean(), 1))
print('Marcus:', round(100 * marcus['score_2026'].mean(), 1))
print()
print('everyone scored 0.35 to 0.65:', len(middle))
print(middle['party'].value_counts())

### Question 6

Katie says the middle range is the only place a knock changes whether someone votes. Can you check that with this voter file? Answer in two sentences: yes or no, and if not, what you would need to know and how you would find it out.

**Your answer:**

*Replace this text with your answer.*

### The memo

Write **250–350 words** to Rachel, your campaign manager. She has ten minutes and a printer deadline.

Your memo must:

1. **Open with the recommendation.** Which 9,000 doors. Say it in the first sentence.
2. **Give your reasons**, using at least two numbers you computed above.
3. **Steelman the strongest objection to your own recommendation**, and answer it. Rachel wants the 9,000 highest scores, Marcus the 9,000 lowest, and Katie the middle (see the case brief). Pick whichever argument cuts hardest against you.
4. **Name what would change your mind.** One number, where you would get it, and which way it would have to come out.

I am grading whether the memo is finished, not whether I agree with it. A memo with no counter-argument is not finished.

**Your memo:**

*Replace this text with your memo.*

---

**Due at 4:00pm on Wednesday Nov 18**, to the **problem set** assignment on Canvas. Whatever you had at 6:00pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Check that every Question 1 to 6 and the memo has an answer.** No cell should still say *Replace this text*.
5. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `[ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full.
6. Upload the PDF to Canvas.